In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# ⚔️ Battle Card Generator: AI-Powered Sales Enablement Agent

An AI agent that instantly generates professional, ready-to-use sales battle cards for any competitor — just enter a name and let the agent do the research.

---

## 🧭 Problem Statement & Vision

Sales reps lose deals not because they lack talent — but because they walk into competitive situations unprepared. Researching a competitor, understanding their weaknesses, crafting objection responses, and building a pitch takes hours of manual work that most reps simply don't have time for.

The average sales team competes against 5-10 competitors regularly, yet battle cards are often outdated, generic, or nonexistent. The result: inconsistent messaging, lost deals, and wasted pipeline.

### The Vision
Battle Card Generator automates this entirely. A sales rep types a competitor's name, and the agent returns a complete, structured battle card in seconds — researched, analyzed, and formatted for immediate use in the field.

---

## 🏗️ Technical Architecture

The agent follows a focused single-orchestrator pipeline with modular tools:

- **MCP Web Search Tool**: Discovers and scrapes competitor information automatically
- **Gemini AI Agent**: Analyzes competitor data and generates the structured battle card
- **Security Layer**: API keys stored in Kaggle Secrets, never exposed in code

## 🔧 Step 1: Setup & Security

We install the required packages and load our API key securely using Kaggle Secrets — 
the key is never visible in the code itself, satisfying security best practices for agent development.

In [1]:
# ============================================================
# SETUP & SECURITY
# API key is loaded from Kaggle Secrets — never hardcoded.
# This follows the security pillar of agentic engineering:
# zero ambient authority, least-privilege credential access.
# ============================================================

# Install required packages
!pip install google-genai requests beautifulsoup4 -q

# Load API key securely from Kaggle Secrets
from kaggle_secrets import UserSecretsClient
from google import genai

secret_value = UserSecretsClient().get_secret("GOOGLE_API_KEY")
client = genai.Client(api_key=secret_value)

print("✅ Setup complete! API key loaded securely.")

✅ Setup complete! API key loaded securely.


## 🔌 Step 2: MCP Web Search Tool

This tool follows the Model Context Protocol (MCP) pattern — a modular, reusable connector 
that the agent calls to research competitors. Instead of hardcoding a URL, the agent 
discovers competitor information dynamically by name, just like a real MCP tool would.

The tool:
1. Searches for the competitor by name
2. Finds their official website
3. Scrapes and cleans the content for the agent to analyze

In [2]:
# ============================================================
# MCP WEB SEARCH TOOL
# Modular tool following MCP pattern: Discover > Configure > Connect
# The agent calls this tool to research any competitor by name.
# Kept separate from the agent logic for reusability.
# ============================================================

import requests
import urllib.parse
from bs4 import BeautifulSoup

def mcp_web_search(competitor_name):
    """
    MCP-style tool: researches a competitor by name.
    Discovers their web presence and returns cleaned content
    for the agent to analyze.
    """
    print(f"🔌 MCP Tool: Searching for '{competitor_name}'...")
    
    search_url = f"https://www.google.com/search?q={urllib.parse.quote(competitor_name + ' product features pricing')}"
    headers = {"User-Agent": "Mozilla/5.0"}
    response = requests.get(search_url, headers=headers, timeout=10)
    soup = BeautifulSoup(response.text, "html.parser")
    
    # Extract competitor's actual website from search results
    links = []
    for a in soup.find_all('a', href=True):
        href = a['href']
        if '/url?q=' in href:
            actual_url = href.split('/url?q=')[1].split('&')[0]
            if competitor_name.lower().replace(' ', '') in actual_url.lower():
                links.append(actual_url)
    
    if links:
        print(f"✅ MCP found competitor URL: {links[0]}")
        # Scrape the competitor's page
        try:
            page = requests.get(links[0], headers=headers, timeout=10)
            page_soup = BeautifulSoup(page.text, "html.parser")
            for tag in page_soup(["nav", "footer", "script", "style"]):
                tag.decompose()
            return page_soup.get_text(separator=" ", strip=True)[:3000]
        except:
            pass
    
    # Fallback: use Google search text directly
    print("⚠️ Using search results as fallback")
    for tag in soup(["nav", "footer", "script", "style"]):
        tag.decompose()
    return soup.get_text(separator=" ", strip=True)[:3000]

print("✅ MCP web search tool ready!")

✅ MCP web search tool ready!


## 🧠 Step 3: Battle Card Agent

This is the brain of the system — a Gemini-powered AI agent that takes the competitor 
research from the MCP tool and generates a structured, actionable battle card.

The agent is given a clear role, structured output format, and your product's key strengths 
so it can generate meaningful differentiators and objection responses — not just generic analysis.

In [3]:
# ============================================================
# BATTLE CARD AGENT
# Orchestrates the full pipeline:
# 1. Calls MCP tool to research the competitor
# 2. Passes research to Gemini with a structured prompt
# 3. Returns a formatted, ready-to-use battle card
# Agent design follows single-responsibility principle —
# one agent, one job, modular tools.
# ============================================================

from IPython.display import Markdown, display

def generate_battle_card(competitor_name, your_product_name, your_key_strengths):
    """
    Main agent pipeline:
    Step 1 — MCP tool researches the competitor
    Step 2 — Gemini agent analyzes and generates the battle card
    """
    
    print(f"🤖 Agent starting research on: {competitor_name}")
    print("─" * 50)
    
    # Step 1: MCP tool call
    competitor_info = mcp_web_search(competitor_name)
    
    # Step 2: Gemini agent generates battle card
    print(f"🧠 Analyzing and generating battle card...")
    
    prompt = f"""
    You are an expert sales enablement agent. Analyze this competitor information 
    and generate a professional sales battle card.
    
    COMPETITOR: {competitor_name}
    COMPETITOR RESEARCH: {competitor_info}
    
    OUR PRODUCT: {your_product_name}
    OUR KEY STRENGTHS: {your_key_strengths}
    
    Generate a battle card with these exact sections:
    
    ## 🏢 Competitor Overview
    (2-3 sentence summary of who they are)
    
    ## 💪 Their Strengths
    (3-4 bullet points)
    
    ## ⚠️ Their Weaknesses
    (3-4 bullet points)
    
    ## 🏆 Our Winning Differentiators
    (3-4 bullet points of why we win)
    
    ## 🗣️ Common Objections & Responses
    (3 objections they raise + our killer responses)
    
    ## ⚡ Quick Pitch When You Meet Them
    (2-3 sentence elevator pitch to use in the field)
    """
    
    response = client.models.generate_content(
        model="gemini-3.1-flash-lite",
        contents=prompt
    )
    
    return response.text

print("✅ Battle card agent ready!")

✅ Battle card agent ready!


## 🚀 Step 4: Generate Your Battle Card

Enter any competitor name and your product details below. 
The agent will automatically research the competitor and generate 
a complete, ready-to-use battle card in seconds.

Simply update the three variables and run the cell:
- `competitor_name` — the competitor you want to research
- `your_product_name` — your product or company name
- `your_key_strengths` — what makes you better (2-3 sentences)

In [4]:
# ============================================================
# RUN THE AGENT
# Update the three variables below and run this cell.
# The agent handles everything else automatically.
# ============================================================

# 👇 EDIT THESE THREE LINES
competitor_name = "HubSpot"
your_product_name = "Acme Sales Pro"
your_key_strengths = "Easy to use, affordable pricing, better customer support, faster onboarding"

# Run the agent
battle_card = generate_battle_card(
    competitor_name=competitor_name,
    your_product_name=your_product_name,
    your_key_strengths=your_key_strengths
)

# Display formatted output
display(Markdown(f"# ⚔️ BATTLE CARD vs {competitor_name}\n\n" + battle_card))

🤖 Agent starting research on: HubSpot
──────────────────────────────────────────────────
🔌 MCP Tool: Searching for 'HubSpot'...
⚠️ Using search results as fallback
🧠 Analyzing and generating battle card...


# ⚔️ BATTLE CARD vs HubSpot

This sales battle card is designed to help you position **Acme Sales Pro** effectively against HubSpot.

*Note: The research provided contained Google consent policy text rather than specific competitor data; therefore, this battle card is populated based on industry-standard market intelligence regarding HubSpot's typical positioning.*

***

## 🏢 Competitor Overview
HubSpot is a large, enterprise-grade CRM and marketing automation platform known for its "all-in-one" ecosystem. They target a broad range of businesses, offering extensive feature sets that cater to complex marketing, sales, and service operations.

## 💪 Their Strengths
* **Ecosystem Depth:** Offers a massive suite of integrated tools (Marketing, Sales, Service, CMS, and Operations Hub).
* **Brand Authority:** Significant market share and a very strong reputation as an industry leader.
* **Content/Education:** World-class resources, including the HubSpot Academy, which drives strong brand loyalty.
* **Marketplace:** A massive library of third-party integrations and app marketplace options.

## ⚠️ Their Weaknesses
* **Complexity & Bloat:** Many users find the platform overwhelming, often requiring expensive external consultants for setup.
* **Aggressive Pricing Tiers:** Costs scale rapidly as you add seats or records; "hidden" costs for advanced features frequently surprise customers.
* **Support Bottlenecks:** Due to their size, their support can feel impersonal and slow, especially for mid-market customers.
* **Onboarding Friction:** Long implementation times mean "time-to-value" is often delayed by months.

## 🏆 Our Winning Differentiators
* **Faster Time-to-Value:** While they require months of setup, Acme Sales Pro is designed for immediate deployment and instant team productivity.
* **Intuitive User Experience:** We prioritize a cleaner, easier-to-use interface that doesn't require a dedicated administrator to manage.
* **Predictable & Affordable Pricing:** No "scale-shock." Our transparent pricing ensures you know your budget up front without unexpected upcharges.
* **High-Touch Support:** You aren't just a ticket number to us; we provide dedicated, accessible support to ensure your team actually uses the software.

## 🗣️ Common Objections & Responses
* **Objection 1: "HubSpot has more features and integrations."**
    * *Response:* "HubSpot has a lot of tools, but are you using all of them? Often, companies pay for a 'Ferrari' but only have the time and training to drive it in first gear. Acme Sales Pro focuses on the features you *actually* use, making your team faster and more efficient without the clutter."
* **Objection 2: "We’re worried about outgrowing your software."**
    * *Response:* "Acme Sales Pro is built to scale with your core revenue processes. We prioritize the fundamentals that actually drive growth rather than complexity that slows your reps down. If you ever need enterprise-level complexity, we’ll be here, but most of our customers find they never needed the 'bloat' of the larger platforms in the first place."
* **Objection 3: "Everybody knows HubSpot; it’s the safe, standard choice."**
    * *Response:* "HubSpot is certainly popular, but 'standard' often means 'harder to use.' If you want your team to actually adopt the CRM and hit their quotas this quarter, you need a tool they *want* to use. Acme Sales Pro offers better adoption rates and faster results because we remove the friction that kills CRM usage."

## ⚡ Quick Pitch When You Meet Them
"While HubSpot is a powerful platform, it often becomes a heavy, expensive, and complex system that creates more work for your team than it saves. Acme Sales Pro gives you the powerful sales tools you need without the bloat, offering faster onboarding, superior support, and a much more intuitive experience that actually drives team adoption from day one."